# Demo 03 : Middleware hooks

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · middleware-hooks

We observe the agent lifecycle and the model and tool calls.

In [2]:
import json
import sqlite3

from langchain.tools import tool

DB = data("demo-world/service.db")


@tool
def bus_info(fleet_no: str) -> str:
    """Look up a Veenhaven bus model and depot by its fleet number."""
    with sqlite3.connect(DB) as conn:
        row = conn.execute(
            "SELECT model, depot_id FROM buses WHERE fleet_no = ?", (fleet_no,)
        ).fetchone()
    return json.dumps(dict(zip(["model", "depot_id"], row))) if row else "Unknown bus"


@tool
def parts_for_model(model: str) -> str:
    """List spare parts, stock and storage depots for a bus model such as CE-12."""
    with sqlite3.connect(DB) as conn:
        rows = conn.execute(
            "SELECT part_no, name, stock, depot FROM parts WHERE model = ?", (model,)
        ).fetchall()
    return json.dumps([dict(zip(["part_no", "name", "stock", "depot"], row)) for row in rows])


tools = [bus_info, parts_for_model]
question = {"messages": [("human", "Which door actuator fits bus CE12-2021-0117, and is it in stock? Look up the bus first.")]}

We log the lifecycle and each model and tool call. Hooks observe or update state, while wrappers control whether the handler runs.

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, wrap_tool_call


class Logger(AgentMiddleware):
    def before_agent(self, state, runtime):
        print("agent: start")

    def before_model(self, state, runtime):
        print("model input messages:", len(state["messages"]))

    def after_model(self, state, runtime):
        print("requested tools:", [c["name"] for c in state["messages"][-1].tool_calls])

    def after_agent(self, state, runtime):
        print("agent: end")


@wrap_tool_call
def log_tool(request, handler):
    print("tool:", request.tool_call["name"], request.tool_call["args"])
    result = handler(request)
    print("result:", result.content)
    return result


agent = create_agent(llm, tools=tools, middleware=[Logger(), log_tool],
                     system_prompt="Look up the bus before its parts. Use the database for stock.")
out = agent.invoke(question, {"recursion_limit": 16})
print("answer:", out["messages"][-1].content)

agent: start
model input messages: 1


requested tools: ['bus_info']
tool: bus_info {'fleet_no': 'CE12-2021-0117'}
result: {"model": "CE-12", "depot_id": "D01"}
model input messages: 3


requested tools: ['parts_for_model']
tool: parts_for_model {'model': 'CE-12'}
result: [{"part_no": "CE-TB-01", "name": "traction battery module", "stock": 1, "depot": "Rietpoort"}, {"part_no": "CE-CF-02", "name": "traction cooling fan", "stock": 6, "depot": "Rietpoort"}, {"part_no": "CE-DA-03", "name": "door actuator", "stock": 0, "depot": "Rietpoort"}, {"part_no": "CE-IM-04", "name": "inverter module", "stock": 2, "depot": "Broekdam"}]
model input messages: 5


requested tools: []
agent: end
answer: Bus **CE12-2021-0117** is a **CE-12**. Its door actuator is part **CE-DA-03**. It is **not in stock** (0 available); the listed storage depot is **Rietpoort**.


We keep logging outside the prompt. Logs can contain identifiers and tool data, so we limit access and redact sensitive fields before recording real traffic.

## Things to try

Optional : change the bus or part identifier and run the example again.